# 02 · 模型训练实验记录
**项目**: 工业表面缺陷检测 | **模型**: YOLOv8s

本 Notebook 记录完整训练过程：
1. 训练配置说明
2. 模型结构分析（参数量/FLOPs）
3. 超参数对比实验（学习率 / 模型尺寸 / 数据增强）
4. 训练曲线可视化
5. 训练 vs 验证 loss 分析

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import pandas as pd
from pathlib import Path

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 120

Path('../results/demo').mkdir(parents=True, exist_ok=True)
print('环境加载完成 ✓')

## 1. 训练配置

In [ ]:
import yaml

config_path = '../configs/train_config.yaml'
with open(config_path) as f:
    config = yaml.safe_load(f)

print('=== 训练配置 ===')
for section, params in config.items():
    print(f'\n[{section}]')
    if isinstance(params, dict):
        for k, v in params.items():
            print(f'  {k}: {v}')
    else:
        print(f'  {params}')

## 2. 模型结构分析

In [ ]:
# 各尺寸 YOLOv8 参数量对比
model_stats = {
    'YOLOv8n': {'params': 3.2, 'flops': 8.7, 'map50': 0.731, 'speed_ms': 1.47},
    'YOLOv8s': {'params': 11.2, 'flops': 28.6, 'map50': 0.783, 'speed_ms': 2.41},
    'YOLOv8m': {'params': 25.9, 'flops': 78.9, 'map50': 0.812, 'speed_ms': 5.09},
    'YOLOv8l': {'params': 43.7, 'flops': 165.2, 'map50': 0.829, 'speed_ms': 7.54},
    'YOLOv8x': {'params': 68.2, 'flops': 257.8, 'map50': 0.841, 'speed_ms': 13.27},
}

df_models = pd.DataFrame(model_stats).T
df_models.columns = ['参数量(M)', 'FLOPs(G)', 'COCO mAP50', '推理速度(ms)']
df_models.index.name = '模型'
print(df_models.to_string())
print('\n→ 选择 YOLOv8s：精度与速度的最佳平衡点（标注 ↑）')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

models = list(model_stats.keys())
params = [model_stats[m]['params'] for m in models]
maps = [model_stats[m]['map50'] for m in models]
speeds = [model_stats[m]['speed_ms'] for m in models]
colors = ['#AAAAAA', '#2196F3', '#AAAAAA', '#AAAAAA', '#AAAAAA']  # 蓝色高亮我们选的

# 参数量 vs mAP
scatter = axes[0].scatter(params, maps, s=[p*15 for p in params],
                           c=colors, edgecolors='black', linewidth=1.5, zorder=3)
for m, p, mp in zip(models, params, maps):
    axes[0].annotate(m, (p, mp), textcoords='offset points', xytext=(8, 4), fontsize=9)
axes[0].set_xlabel('参数量 (M)', fontsize=11)
axes[0].set_ylabel('COCO mAP50', fontsize=11)
axes[0].set_title('模型参数量 vs 精度', fontsize=12, fontweight='bold')
axes[0].grid(True, alpha=0.3)

# 速度 vs mAP
axes[1].scatter(speeds, maps, s=[p*15 for p in params],
                c=colors, edgecolors='black', linewidth=1.5, zorder=3)
for m, s, mp in zip(models, speeds, maps):
    axes[1].annotate(m, (s, mp), textcoords='offset points', xytext=(8, 4), fontsize=9)
axes[1].set_xlabel('推理速度 (ms/img)', fontsize=11)
axes[1].set_ylabel('COCO mAP50', fontsize=11)
axes[1].set_title('推理速度 vs 精度', fontsize=12, fontweight='bold')
axes[1].grid(True, alpha=0.3)
axes[1].text(2.5, 0.786, '← 本项目选择', fontsize=10, color='#2196F3', fontweight='bold')

plt.suptitle('YOLOv8 各尺寸模型对比（蓝点为本项目所选）', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/demo/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ 已保存: results/demo/model_comparison.png')

## 3. 超参数消融实验

In [ ]:
# 超参数消融实验结果（模拟训练结果，真实训练后替换）
ablation_data = {
    '实验': [
        'Baseline (lr=1e-3)',
        'lr=5e-4',
        'lr=2e-4 ✓',
        'lr=1e-4',
        'No Augment',
        'Augment (弱)',
        'Augment (强) ✓',
        'Mosaic Off',
        'Mosaic On ✓',
    ],
    'mAP50': [0.741, 0.769, 0.821, 0.803, 0.698, 0.774, 0.821, 0.789, 0.821],
    'mAP50-95': [0.412, 0.438, 0.487, 0.471, 0.381, 0.443, 0.487, 0.451, 0.487],
    'Precision': [0.801, 0.823, 0.856, 0.841, 0.762, 0.831, 0.856, 0.837, 0.856],
    'Recall': [0.712, 0.738, 0.789, 0.771, 0.681, 0.744, 0.789, 0.758, 0.789],
}
df_ablation = pd.DataFrame(ablation_data)
df_ablation = df_ablation.set_index('实验')
print(df_ablation.to_string())
print('\n→ 最优配置：lr=2e-4 + 强数据增强 + Mosaic')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# 学习率消融
lr_exps = ['1e-3', '5e-4', '2e-4\n(最优)', '1e-4']
lr_maps = [0.741, 0.769, 0.821, 0.803]
bar_colors = ['#BBBBBB', '#BBBBBB', '#2196F3', '#BBBBBB']
bars = axes[0].bar(lr_exps, lr_maps, color=bar_colors, edgecolor='white', linewidth=1.5)
for bar, val in zip(bars, lr_maps):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
                 f'{val:.3f}', ha='center', fontsize=10, fontweight='bold')
axes[0].set_ylim(0.68, 0.87)
axes[0].set_xlabel('学习率', fontsize=11)
axes[0].set_ylabel('mAP50', fontsize=11)
axes[0].set_title('学习率消融实验', fontsize=12, fontweight='bold')

# 数据增强消融
aug_exps = ['无增强', '弱增强', '强增强\n(最优)', 'Mosaic Off', 'Mosaic On\n(最优)']
aug_maps = [0.698, 0.774, 0.821, 0.789, 0.821]
aug_colors = ['#BBBBBB', '#BBBBBB', '#2196F3', '#BBBBBB', '#2196F3']
bars2 = axes[1].bar(aug_exps, aug_maps, color=aug_colors, edgecolor='white', linewidth=1.5)
for bar, val in zip(bars2, aug_maps):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
                 f'{val:.3f}', ha='center', fontsize=10, fontweight='bold')
axes[1].set_ylim(0.64, 0.87)
axes[1].set_xlabel('增强策略', fontsize=11)
axes[1].set_ylabel('mAP50', fontsize=11)
axes[1].set_title('数据增强消融实验', fontsize=12, fontweight='bold')

plt.suptitle('超参数消融实验结果（蓝色=最优配置）', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/demo/ablation_study.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ 已保存: results/demo/ablation_study.png')

## 4. 训练曲线可视化

In [ ]:
# 模拟训练日志（真实训练后可从 runs/detect/*/results.csv 读取）
def simulate_training_curve(epochs=100, seed=42):
    np.random.seed(seed)
    e = np.arange(1, epochs + 1)

    # Box loss
    train_box = 1.8 * np.exp(-e / 30) + 0.35 + np.random.normal(0, 0.015, epochs)
    val_box   = 1.9 * np.exp(-e / 28) + 0.38 + np.random.normal(0, 0.025, epochs)

    # Cls loss
    train_cls = 2.1 * np.exp(-e / 25) + 0.28 + np.random.normal(0, 0.012, epochs)
    val_cls   = 2.2 * np.exp(-e / 23) + 0.31 + np.random.normal(0, 0.022, epochs)

    # mAP50
    map50 = 0.82 * (1 - np.exp(-e / 35)) + np.random.normal(0, 0.008, epochs)
    map50 = np.clip(map50, 0, 1)

    # Precision / Recall
    prec = 0.856 * (1 - np.exp(-e / 30)) + np.random.normal(0, 0.010, epochs)
    rec  = 0.789 * (1 - np.exp(-e / 38)) + np.random.normal(0, 0.010, epochs)

    return pd.DataFrame({
        'epoch': e,
        'train/box_loss': np.clip(train_box, 0.3, 3),
        'val/box_loss': np.clip(val_box, 0.35, 3.2),
        'train/cls_loss': np.clip(train_cls, 0.25, 3),
        'val/cls_loss': np.clip(val_cls, 0.28, 3.2),
        'metrics/mAP50': map50,
        'metrics/precision': np.clip(prec, 0, 1),
        'metrics/recall': np.clip(rec, 0, 1),
    })

df_log = simulate_training_curve(epochs=100)

# 尝试读取真实日志（如果存在）
real_log = Path('../runs/detect/defect_v1/results.csv')
if real_log.exists():
    df_log = pd.read_csv(real_log)
    df_log.columns = df_log.columns.str.strip()
    print(f'✓ 使用真实训练日志: {real_log}')
else:
    print('→ 使用模拟训练曲线（真实训练完成后替换）')

print(f'  总 epoch: {len(df_log)}')
print(f'  最终 mAP50: {df_log["metrics/mAP50"].iloc[-1]:.4f}')

In [ ]:
fig = plt.figure(figsize=(16, 10))
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)

epochs = df_log['epoch']

# Box Loss
ax1 = fig.add_subplot(gs[0, 0])
ax1.plot(epochs, df_log['train/box_loss'], label='Train', color='#2196F3', linewidth=1.5)
ax1.plot(epochs, df_log['val/box_loss'], label='Val', color='#FF5722', linewidth=1.5, linestyle='--')
ax1.set_title('Box Loss', fontweight='bold')
ax1.set_xlabel('Epoch')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Cls Loss
ax2 = fig.add_subplot(gs[0, 1])
ax2.plot(epochs, df_log['train/cls_loss'], label='Train', color='#2196F3', linewidth=1.5)
ax2.plot(epochs, df_log['val/cls_loss'], label='Val', color='#FF5722', linewidth=1.5, linestyle='--')
ax2.set_title('Classification Loss', fontweight='bold')
ax2.set_xlabel('Epoch')
ax2.legend()
ax2.grid(True, alpha=0.3)

# mAP50
ax3 = fig.add_subplot(gs[0, 2])
ax3.plot(epochs, df_log['metrics/mAP50'], color='#4CAF50', linewidth=2)
best_epoch = df_log['metrics/mAP50'].idxmax()
best_map = df_log['metrics/mAP50'].max()
ax3.scatter(df_log['epoch'].iloc[best_epoch], best_map,
            color='red', zorder=5, s=100, label=f'Best: {best_map:.4f} (ep{df_log["epoch"].iloc[best_epoch]:.0f})')
ax3.set_title('mAP50', fontweight='bold')
ax3.set_xlabel('Epoch')
ax3.legend(fontsize=8)
ax3.grid(True, alpha=0.3)

# Precision
ax4 = fig.add_subplot(gs[1, 0])
ax4.plot(epochs, df_log['metrics/precision'], color='#9C27B0', linewidth=1.5)
ax4.set_title('Precision', fontweight='bold')
ax4.set_xlabel('Epoch')
ax4.set_ylim(0, 1.05)
ax4.grid(True, alpha=0.3)

# Recall
ax5 = fig.add_subplot(gs[1, 1])
ax5.plot(epochs, df_log['metrics/recall'], color='#FF9800', linewidth=1.5)
ax5.set_title('Recall', fontweight='bold')
ax5.set_xlabel('Epoch')
ax5.set_ylim(0, 1.05)
ax5.grid(True, alpha=0.3)

# P-R 曲线
ax6 = fig.add_subplot(gs[1, 2])
prec_vals = df_log['metrics/precision'].values
rec_vals = df_log['metrics/recall'].values
sorted_idx = np.argsort(rec_vals)
ax6.plot(rec_vals[sorted_idx], prec_vals[sorted_idx], color='#2196F3', linewidth=2)
ax6.fill_between(rec_vals[sorted_idx], prec_vals[sorted_idx], alpha=0.2, color='#2196F3')
ax6.set_title('Precision-Recall 曲线', fontweight='bold')
ax6.set_xlabel('Recall')
ax6.set_ylabel('Precision')
ax6.set_xlim(0, 1)
ax6.set_ylim(0, 1.05)
ax6.grid(True, alpha=0.3)

fig.suptitle('YOLOv8s 训练过程完整记录', fontsize=16, fontweight='bold')
plt.savefig('../results/demo/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ 已保存: results/demo/training_curves.png')

## 5. 训练总结

| 配置项 | 数值 |
|--------|------|
| 模型 | YOLOv8s |
| 参数量 | 11.2M |
| 训练 Epoch | 100 |
| 学习率 | 2e-4 (cosine decay) |
| Batch Size | 16 |
| 最优 mAP50 | ~0.82 |
| Early Stopping | patience=20 |

> **下一步**: 进入 `03_results_analysis.ipynb` 进行详细结果分析